In [16]:
"""# Hydroclimatic Pattern Discovery – Gatun Lake

## 01. Data Collection

**Objective:** Retrieve historical meteorological data from NASA POWER API for a geographic reference point near Gatun Lake, Panama.

**Data source:** NASA Prediction Of Worldwide Energy Resources (POWER).

**Study period:** January 2015 – December 2025.

**Machine Learning approach:** Unsupervised Learning.

**Algorithms:** Principal Component Analysis (PCA) and K-Means Clustering.

**Research question:** Can dimensionality reduction preserve sufficient information to identify meaningful hydroclimatic patterns?"""

'# Hydroclimatic Pattern Discovery – Gatun Lake\n\n## 01. Data Collection\n\n**Objective:** Retrieve historical meteorological data from NASA POWER API for a geographic reference point near Gatun Lake, Panama.\n\n**Data source:** NASA Prediction Of Worldwide Energy Resources (POWER).\n\n**Study period:** January 2015 – December 2025.\n\n**Machine Learning approach:** Unsupervised Learning.\n\n**Algorithms:** Principal Component Analysis (PCA) and K-Means Clustering.\n\n**Research question:** Can dimensionality reduction preserve sufficient information to identify meaningful hydroclimatic patterns?'

In [17]:
# 01 - Importación de Librerias 

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt 
import seaborn as sns 

print("Librerias importadas")
print("Versión de pandas:", pd.__version__)
print("Versión de numpy:", np.__version__)

Librerias importadas
Versión de pandas: 3.1.0
Versión de numpy: 2.5.4


In [18]:
# 02 - Configuración geográfica y temporal

LATITUDE = 9.20
LONGITUDE = -79.90

START_DATE = "20150101"
END_DATE = "20251231"

print("Latitud", LATITUDE)
print("Longitude", LONGITUDE)
print("Start date", START_DATE)
print("End date", END_DATE)


Latitud 9.2
Longitude -79.9
Start date 20150101
End date 20251231


In [19]:
# 03 - Variables Meteorológicas

PARAMETERS = [
    "PRECTOTCORR",
    "T2M",
    "RH2M",
    "WS10M",
    "ALLSKY_SFC_SW_DWN"
]

print("Número de variables:", len(PARAMETERS))

for parameter in PARAMETERS:
    print(parameter)


Número de variables: 5
PRECTOTCORR
T2M
RH2M
WS10M
ALLSKY_SFC_SW_DWN


In [20]:
# 04 - Configuraciónd de la API, descarga de datos y formato

BASE_URL = (
    "https://power.larc.nasa.gov/"
    "api/temporal/daily/point"
)

api_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start": START_DATE,
    "end": END_DATE,
    "parameters": ",".join(PARAMETERS),
    "community": "AG",
    "format": "JSON",
    "time-standard": "LST"
}

print("URL base:", BASE_URL)
print("Parámetros:", api_params)

URL base: https://power.larc.nasa.gov/api/temporal/daily/point
Parámetros: {'latitude': 9.2, 'longitude': -79.9, 'start': '20150101', 'end': '20251231', 'parameters': 'PRECTOTCORR,T2M,RH2M,WS10M,ALLSKY_SFC_SW_DWN', 'community': 'AG', 'format': 'JSON', 'time-standard': 'LST'}


In [21]:
# 05 - Solicitud HTTP

response = requests.get(
    BASE_URL,
    params=api_params,
    timeout=120
)

print("Código HTTP:", response.status_code)
print("URL Solicitada:", response.url)

response.raise_for_status()

print("Solicitud completada correctamente")


Código HTTP: 200
URL Solicitada: https://power.larc.nasa.gov/api/temporal/daily/point?latitude=9.2&longitude=-79.9&start=20150101&end=20251231&parameters=PRECTOTCORR%2CT2M%2CRH2M%2CWS10M%2CALLSKY_SFC_SW_DWN&community=AG&format=JSON&time-standard=LST
Solicitud completada correctamente


In [22]:
# 06 - Interpretación de JSON

json_data = response.json()

print("Claves principales:")
print(list(json_data.keys()))

print("\nClaves de properties:")
print(list(json_data["properties"].keys()))


Claves principales:
['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times']

Claves de properties:
['parameter']


In [23]:
# 07 - Convirtiendo de JSON a Dataframe 
waether_data = json_data["properties"]["parameter"]

df = pd.DataFrame(waether_data)

print("Dimensiones:", df.shape)
display(df.head())

Dimensiones: (4018, 5)


,PRECTOTCORR,T2M,RH2M,WS10M,ALLSKY_SFC_SW_DWN
20150101,0.21,26.13,80.61,4.58,17.01
20150102,0.11,26.67,81.68,4.60,15.97
20150103,0.15,27.05,82.74,3.96,17.29
20150104,0.44,26.72,81.27,4.31,16.27
20150105,0.15,26.57,81.43,4.91,17.97


In [24]:
# 08 - Convirtiendo las fechas a DatetimeIndex

# Tratamiento del indice temporal (tiempo)

df.index = pd.to_datetime(
    df.index,
    format="%Y%m%d"
)

df.index_name = "date"

df = df.sort_index()

print("Tipo de indice:", type(df.index))
print("Fecha inicial:", df.index.min())
print("Fecha final:", df.index.max())

display(df.head())


Tipo de indice: <class 'pandas.DatetimeIndex'>
Fecha inicial: 2015-01-01 00:00:00
Fecha final: 2025-12-31 00:00:00


,PRECTOTCORR,T2M,RH2M,WS10M,ALLSKY_SFC_SW_DWN
2015-01-01,0.21,26.13,80.61,4.58,17.01
2015-01-02,0.11,26.67,81.68,4.60,15.97
2015-01-03,0.15,27.05,82.74,3.96,17.29
2015-01-04,0.44,26.72,81.27,4.31,16.27
2015-01-05,0.15,26.57,81.43,4.91,17.97


In [25]:
# 09 - Validación de la información recibida sobre lo que se solicito

expected_columns = set(PARAMETERS)
received_columns = set(df.columns)

missing_columns = expected_columns - received_columns

print("Filas:", len(df))
print("Columnas:", len(df.columns))
print("Columnas faltantes:", missing_columns)
print("Fechas duplicadas:", df.index.duplicated().sum())

assert not missing_columns, "Faltan variables solicitadas"
assert df.index.is_unique, "Existen fechas duplicadas"
assert df.index.is_monotonic_increasing, "Fechas desordenadas"

print("Validaciones estructurales superadas")

Filas: 4018
Columnas: 5
Columnas faltantes: set()
Fechas duplicadas: 0
Validaciones estructurales superadas


In [26]:
# 10 - Validar continuidad temporal

expected_dates = pd.date_range(
    start="2015-01-01",
    end="2025-12-31",
    freq="D"
)

missing_dates = expected_dates.difference(df.index)

print("Días esperados:", len(expected_dates))
print("Días recibidos:", len(df))
print("Fechas ausentes:", len(missing_dates))

if len(missing_dates) > 0:
    print(missing_dates[:10])

Días esperados: 4018
Días recibidos: 4018
Fechas ausentes: 0


In [27]:
# 11 - Localizar raíz del proyecto 

from pathlib import Path

current = Path.cwd()

if current.name == "notebooks":
    PROJECT_ROOT = current.parent
else:
    PROJECT_ROOT = current

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raíz:", PROJECT_ROOT)
print("Carpeta de datos:", RAW_DATA_DIR)

Raíz: c:\Users\richa\ACP_Hydroclimatic_Clustering
Carpeta de datos: c:\Users\richa\ACP_Hydroclimatic_Clustering\data\raw


In [28]:
# 12 - Exportando documento CSV Original 

output_file = RAW_DATA_DIR / "nasa_power_gatum_2015_2025.csv"

df.to_csv(
    output_file,
    index=True,
    index_label="date"
)

print("Archivo guardado en:")
print(output_file.resolve())

Archivo guardado en:
C:\Users\richa\ACP_Hydroclimatic_Clustering\data\raw\nasa_power_gatum_2015_2025.csv


In [29]:
# 13 - Comprobando archivo guardado

df_check = pd.read_csv(
    output_file,
    parse_dates=["date"],
    index_col="date"
)

print("Dimensiones originales:", df.shape)
print("Dimensiones recuperadas:", df_check.shape)

assert df.shape == df_check.shape

display(df_check.head())

Dimensiones originales: (4018, 5)
Dimensiones recuperadas: (4018, 5)


,PRECTOTCORR,T2M,RH2M,WS10M,ALLSKY_SFC_SW_DWN
date,,,,,
2015-01-01,0.21,26.13,80.61,4.58,17.01
2015-01-02,0.11,26.67,81.68,4.60,15.97
2015-01-03,0.15,27.05,82.74,3.96,17.29
2015-01-04,0.44,26.72,81.27,4.31,16.27
2015-01-05,0.15,26.57,81.43,4.91,17.97


In [30]:
""" Guardando metadatos de la descarga """

# 14 - Guardar metadatos 

import json
from datetime import datetime, timezone

metadata = {
    "project": "ACP Hydroclimatic Clustering",
    "source": "NASA POWER Daily API",
    "api_url": response.url,
    "Latitude": LATITUDE,
    "Longitude": LONGITUDE,
    "start-date": START_DATE,
    "end-date": END_DATE,
    "parameters": PARAMETERS,
    "time_standard": "LST",
    "downloaded_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "rows": len(df),
    "columns": list(df.columns)
}

metadata_file = RAW_DATA_DIR / "metadata.json"

with open(metadata_file, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4, ensure_ascii=False)

print("Metadatos guardados correctamente")

Metadatos guardados correctamente
